In [ ]:
# MISSING DAYS INTERPOLATION CHECK

import re
import os 
import utilities_file as utils

full_analyzed_dir = r"Z:\People\JenLi\Imaging_Data\L23_L5_structural_imaging\Analyzed_Data"
minimal_test_dir = os.path.join(full_analyzed_dir, "missing_day_test")

mouse_id = "JL117"
fov_id = "FOV3"
dates = ['250211', '250212', '250213', '250214', '250215', '250216', '250217', '250218', '250219', '250220', '250221', '250222', '250224', '250225']
df_list = []

for i, date in enumerate(dates):
    if date == "missing":
        df_list.append(None)
        continue
    date_path = os.path.join(minimal_test_dir, mouse_id, fov_id, date)
    pickle_files = [f for f in os.listdir(date_path) if f.endswith(".pickle")]
    pickle_file = pickle_files[0]

    match = re.search(r"(JL\d+).*_(L\d+)\.pickle", pickle_file) #extract mouse ID & layer
    layer = match.group(2)

    df_out = utils.get_single_day_df(
                                    fname=os.path.join(minimal_test_dir,mouse_id,fov_id,date,pickle_file),
                                    fov_id=fov_id,
                                    mouse_id=mouse_id
                                    )

    # Assign day number and layer ID
    df_out["day"] = i + 1
    df_out["layer"] = layer

    df_list.append(df_out)

interpolate_cols = ["position", "volume"]
for i, df in enumerate(df_list):
    if df is None:
        df_loc = i
        prev_df = df_list[i-1]
        next_df = df_list[i+1]

        # select only spines that exist day 1 & day 3
        shared_spines = prev_df["spine_ID"].isin(next_df["spine_ID"])

        missing_df = prev_df[shared_spines].copy() # missing_df starts with data from day 1
        next_df_shared = next_df[next_df["spine_ID"].isin(missing_df["spine_ID"])].copy()
        missing_df = missing_df.reset_index(drop=True)
        next_df_shared = next_df_shared.reset_index(drop=True)

        missing_df[interpolate_cols] = (
            missing_df[interpolate_cols] + next_df_shared[interpolate_cols]
        ) / 2

        missing_df["day"] = missing_df["day"] + 1 # update day (is day 1 until updated)

        print(f"Missing location:",df_loc)
        df_list[df_loc] = missing_df

In [ ]:
# CHECK INDIVIDUAL PICKLE FILES

import pickle

with open(r"Z:\People\JenLi\Imaging_Data\L23_L5_structural_imaging\Analyzed_Data\test_JL122\JL122\FOV5\250402\JL122_250402_imaging_data_L5.pickle","rb") as f:
    data = pickle.load(f)